In [1]:
import pandas as pd
import numpy as np
#import seaborn as sns
import matplotlib.pyplot as plt

from itertools import chain

from LongTermOptimizer import (
    generate_single_long_term_return,
    estimate_long_term_return,
    minimize_simulation,
    INDEX_TO_SCENARIO
)
from data import (
    load_metadata_artefacts,
    load_odds,
    join_metadata,
    build_empty_dataframe,
    apply_final_treatment,
)
from GameProbs import GameProbs
from dependencies.utils import get_scenarios
from filter import filter_by_linear_combination
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [2]:
metadata, gameid_to_outcome = load_metadata_artefacts("data/metadata-with-date.parquet")
odds = load_odds("data/odds.parquet")
print(metadata.shape)
print(odds.shape)

(16869, 9)
(2003933, 7)


In [3]:
# Join metadata info to the odds dataframe
odds = join_metadata(odds, metadata)
print(odds.shape)

(2003933, 10)


In [4]:
odds = odds[odds.Datetime.apply(str)=='2022-07-09']
odds.GameId.unique()

array(['5925201', '5925200', '5925186'], dtype=object)

In [5]:
GAME_ID = '5925201'
odds_sample = odds[(odds.GameId==GAME_ID)]
#odds_sample = join_metadata(odds_sample, metadata)
games_ids = odds_sample['GameId'].unique()

# Initialize dict to store dataframes of favorable bet opportunities
odds_dict = {}
# Initialize dict to store 7x7 matrices/dataframes of real probabilities 
df_probs_dict = {}

for game_id in games_ids:
    df = GameProbs(game_id).build_dataframe()
    odds_sample = apply_final_treatment(df_odds=odds_sample, df_real_prob=df)
    odds_sample = filter_by_linear_combination(odds_sample)
    odds_dict[game_id] = odds_sample
    df_probs_dict[game_id] = df

In [6]:
n = len(odds_sample)
allocation_array = np.round(np.array(((1/n), ) * n), 4)
allocation_array

array([0.2, 0.2, 0.2, 0.2, 0.2])

In [7]:
df

,0,1,2,3,4,5,6
0,0.0492,0.0766,0.0705,0.0448,0.0222,0.0091,0.0046
1,0.0555,0.1038,0.0848,0.0525,0.0253,0.0102,0.0050
2,0.0383,0.0636,0.0614,0.0337,0.0160,0.0063,0.0031
3,0.0188,0.0305,0.0261,0.0183,0.0073,0.0029,0.0014
4,0.0074,0.0118,0.0099,0.0058,0.0036,0.0010,0.0005
5,0.0025,0.0039,0.0032,0.0019,0.0009,0.0006,0.0002
6,0.0010,0.0015,0.0012,0.0007,0.0003,0.0001,0.0001


In [8]:
df_probs_dict

{'5925201':         0       1       2       3       4       5       6
 0  0.0492  0.0766  0.0705  0.0448  0.0222  0.0091  0.0046
 1  0.0555  0.1038  0.0848  0.0525  0.0253  0.0102  0.0050
 2  0.0383  0.0636  0.0614  0.0337  0.0160  0.0063  0.0031
 3  0.0188  0.0305  0.0261  0.0183  0.0073  0.0029  0.0014
 4  0.0074  0.0118  0.0099  0.0058  0.0036  0.0010  0.0005
 5  0.0025  0.0039  0.0032  0.0019  0.0009  0.0006  0.0002
 6  0.0010  0.0015  0.0012  0.0007  0.0003  0.0001  0.0001}

In [9]:
solution = minimize_simulation(df_prob=df,
                               df_bet=odds_sample,
                               num_simulations=50)

/Users/marcosbarbosa/anaconda3/envs/soccer_betting_strategy_env/lib/python3.9/site-packages/scipy/optimize/_minimize.py:576: RuntimeWarning: Method Powell cannot handle constraints.
  warn('Method %s cannot handle constraints.' % method,


Optimization terminated successfully.
         Current function value: -294971485188346955769881877807104.000000
         Iterations: 4
         Function evaluations: 146


In [10]:
solution

array([0.85322812, 0.97339324, 0.77469108, 0.83509562, 0.82259328])